# Agentic AI basics: a small working agent

This is the companion notebook to the article *Agentic AI in 5 Minutes: The Basics I Needed Before Using It in My Research*.

It builds a small agent step by step, following the same order as the article:

1. **The loop**, first with a pretend model so you can see the mechanics without an API key.
2. **Tools**, written as ordinary Python functions with a short description.
3. **A real agent**, where Claude decides which tools to call.
4. **Guardrails**: a step limit, read-only tools, and a log of every step.

The task is a typical research chore: check a folder of GWAS summary statistics files from a pipeline run, find any files that are missing or empty, and count genome-wide significant hits in the rest.

All data in this notebook is **synthetic** and generated below. No real genetic data is used.

## 0. Setup

You need Python 3.9 or later. Part 1 runs without an API key. Part 3 needs an Anthropic API key, which you can create in the Claude Console at https://platform.claude.com.

Set the key as an environment variable before starting Jupyter, so it never appears in the notebook itself:

```bash
export ANTHROPIC_API_KEY="your-key-here"
```

In [ ]:
%pip install --quiet anthropic pandas numpy

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

## 1. Create a pretend pipeline output

The cell below writes summary statistics for chromosomes 1 to 5 into a `results/` folder. To give the agent something to find, two things are wrong on purpose:

- the file for chromosome 3 is empty, as if the job crashed after creating it;
- the file for chromosome 5 is missing completely.

Chromosome 2 also contains a few SNPs with p-values below 5e-8.

In [ ]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

rng = np.random.default_rng(seed=42)


def make_sumstats(chrom, n_snps=2000, n_hits=0):
    """Return a synthetic summary statistics table for one chromosome."""
    p = rng.uniform(1e-6, 1, size=n_snps)
    p[:n_hits] = rng.uniform(1e-12, 1e-9, size=n_hits)  # plant some significant hits
    return pd.DataFrame({
        "SNP": [f"rs{chrom}{i:06d}" for i in range(n_snps)],
        "CHR": chrom,
        "BP": np.sort(rng.integers(1, 2_000_000, size=n_snps)),
        "BETA": rng.normal(0, 0.02, size=n_snps).round(5),
        "P": p,
    })


for chrom in [1, 2, 4]:
    hits = 4 if chrom == 2 else 0
    make_sumstats(chrom, n_hits=hits).to_csv(RESULTS_DIR / f"sumstats_chr{chrom}.tsv", sep="\t", index=False)

(RESULTS_DIR / "sumstats_chr3.tsv").touch()  # empty file: the "crashed" job
# chromosome 5 is deliberately not written

sorted(p.name for p in RESULTS_DIR.iterdir())

## 2. Tools: what the agent is allowed to do

A tool is a normal Python function. The agent never runs these itself. It asks for one by name, and **our** code runs it.

Two design choices keep this safe:

- every tool is **read-only**, so the agent cannot change or delete anything;
- every tool only looks inside `results/`, so the agent cannot read other files on your computer.

In [ ]:
EXPECTED_CHROMOSOMES = [1, 2, 3, 4, 5]


def _safe_path(filename):
    """Resolve a filename inside RESULTS_DIR and refuse anything outside it."""
    path = (RESULTS_DIR / filename).resolve()
    if RESULTS_DIR.resolve() not in path.parents:
        raise ValueError(f"Access outside {RESULTS_DIR}/ is not allowed: {filename}")
    return path


def list_result_files():
    """List the files in the results folder with their sizes in bytes."""
    return {p.name: p.stat().st_size for p in sorted(RESULTS_DIR.iterdir()) if p.is_file()}


def count_rows(filename):
    """Count the data rows in a summary statistics file (header not counted)."""
    path = _safe_path(filename)
    if path.stat().st_size == 0:
        return {"filename": filename, "rows": 0, "note": "file is empty"}
    return {"filename": filename, "rows": len(pd.read_csv(path, sep="\t"))}


def count_significant(filename, p_threshold=5e-8):
    """Count SNPs below a p-value threshold and return the top few."""
    path = _safe_path(filename)
    if path.stat().st_size == 0:
        return {"filename": filename, "error": "file is empty"}
    df = pd.read_csv(path, sep="\t")
    hits = df[df["P"] < p_threshold].sort_values("P")
    return {
        "filename": filename,
        "p_threshold": p_threshold,
        "n_significant": int(len(hits)),
        "top_hits": hits.head(5)[["SNP", "BP", "P"]].to_dict(orient="records"),
    }


# The registry maps the name the model uses to the function we run.
TOOL_FUNCTIONS = {
    "list_result_files": list_result_files,
    "count_rows": count_rows,
    "count_significant": count_significant,
}

# Try the tools by hand first. It is good practice to test them before an agent uses them.
print(list_result_files())
print(count_rows("sumstats_chr3.tsv"))
print(count_significant("sumstats_chr2.tsv"))

## 3. The loop, with a pretend model

Before using a real model, here is the agent loop with a **scripted** model that always makes the same choices. This runs without an API key and shows the three steps that repeat: **decide, act, observe**.

In [ ]:
class PretendModel:
    """Stands in for an LLM. It replays a fixed list of decisions."""

    def __init__(self):
        self.script = [
            {"tool": "list_result_files", "args": {}},
            {"tool": "count_rows", "args": {"filename": "sumstats_chr3.tsv"}},
            {"tool": "count_significant", "args": {"filename": "sumstats_chr2.tsv"}},
            {"final_answer": "chr3 is empty, chr5 is missing, chr2 has 4 significant hits."},
        ]

    def decide(self, history):
        return self.script[len(history) - 1]


def run_pretend_agent(goal, max_steps=10):
    model = PretendModel()
    history = [goal]

    for step in range(1, max_steps + 1):
        decision = model.decide(history)                        # 1. decide
        if "final_answer" in decision:
            print(f"Step {step}: final answer -> {decision['final_answer']}")
            return decision["final_answer"]

        result = TOOL_FUNCTIONS[decision["tool"]](**decision["args"])  # 2. act
        print(f"Step {step}: {decision['tool']}({decision['args']}) -> {result}")
        history.append(result)                                  # 3. observe

    return "Stopped: step limit reached"


run_pretend_agent("Check the pipeline output.")

The pretend model cannot adapt: if the files were different, it would still make the same calls. A real model reads each result and decides the next step from it. That is the only part we replace in the next section.

## 4. A real agent with Claude

### 4a. Describe the tools to the model

The model cannot see our Python code. It only sees a **name**, a **description**, and an **input schema** for each tool. Clear descriptions lead to better tool choices.

In [ ]:
TOOLS = [
    {
        "name": "list_result_files",
        "description": "List all files in the pipeline results folder, with their sizes in bytes. "
                       "A size of 0 means the file is empty.",
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "count_rows",
        "description": "Count the data rows (SNPs) in one summary statistics file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "File name, for example sumstats_chr1.tsv"},
            },
            "required": ["filename"],
        },
    },
    {
        "name": "count_significant",
        "description": "Count SNPs with a p-value below a threshold in one summary statistics file "
                       "and return the top 5 hits. Default threshold is 5e-8 (genome-wide significance).",
        "input_schema": {
            "type": "object",
            "properties": {
                "filename": {"type": "string", "description": "File name, for example sumstats_chr1.tsv"},
                "p_threshold": {"type": "number", "description": "P-value threshold. Default 5e-8."},
            },
            "required": ["filename"],
        },
    },
]

### 4b. The agent loop

This is the same loop as in Part 3. The only changes are that Claude makes the decisions, and that every step is written to a log list so you can check afterwards what the agent did.

When Claude wants a tool, the API response has `stop_reason == "tool_use"` and one or more `tool_use` blocks. We run each requested tool and send the result back as a `tool_result` block. When Claude has finished, it answers in plain text instead.

In [ ]:
import anthropic

MODEL = "claude-sonnet-5-5"  # see https://platform.claude.com/docs/en/models/overview for current model IDs

SYSTEM_PROMPT = (
    "You are a careful research assistant checking the output of a GWAS pipeline. "
    f"The pipeline should produce one file per chromosome for chromosomes {EXPECTED_CHROMOSOMES}, "
    "named sumstats_chr<N>.tsv. Use the tools to investigate before answering. "
    "Report missing files, empty files, row counts, and genome-wide significant hits. "
    "Only state numbers that come from tool results."
)


def run_tool(name, args):
    """Run one tool requested by the model. Errors go back to the model as text."""
    try:
        return json.dumps(TOOL_FUNCTIONS[name](**args)), False
    except Exception as exc:  # the model sees the error and can try something else
        return f"Error: {exc}", True


def run_agent(goal, client, max_steps=15):
    messages = [{"role": "user", "content": goal}]
    log = []

    for step in range(1, max_steps + 1):
        # 1. decide: the model looks at everything so far
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            answer = "".join(block.text for block in response.content if block.type == "text")
            return answer, log

        # 2. act: run every tool the model asked for in this turn
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            output, is_error = run_tool(block.name, block.input)
            log.append({"step": step, "tool": block.name, "input": block.input, "output": output})
            print(f"Step {step}: {block.name}({block.input})")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": output,
                "is_error": is_error,
            })

        # 3. observe: the results go back to the model for the next step
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: step limit reached", log

### 4c. Run it

The goal is written the way you would brief a colleague. Notice that it does not say which files are broken; the agent has to find that out.

In [ ]:
goal = (
    "Please check last night's pipeline run in the results folder. "
    "Are all expected files there and non-empty? For the good files, how many SNPs "
    "does each have, and are there any genome-wide significant hits?"
)

if os.environ.get("ANTHROPIC_API_KEY"):
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    answer, log = run_agent(goal, client)
    print("\n--- Agent's report ---\n")
    print(answer)
else:
    print("ANTHROPIC_API_KEY is not set, so this cell was skipped. "
          "Set the key and restart Jupyter to run the real agent.")

### 4d. Check the agent's work

An agent's answer should be checked like any other analysis. The log shows every tool call and its raw output, so you can confirm that the numbers in the report came from the data.

In [ ]:
if "log" in globals():
    display(pd.DataFrame(log)[["step", "tool", "input", "output"]])

## 5. Guardrails used in this notebook

| Guardrail | Where | Why |
|---|---|---|
| Step limit | `max_steps` in `run_agent` | Stops the agent if it loops without finishing |
| Read-only tools | Section 2 | The agent can look at files but not change them |
| Folder restriction | `_safe_path` | The agent cannot read anything outside `results/` |
| Errors returned as text | `run_tool` | The agent can recover from a bad call instead of crashing the loop |
| Step log | `log` list | Every action can be checked afterwards |
| Synthetic data | Section 1 | No individual-level data is sent to an external API |

If you adapt this to real data, check your institution's rules first. Summary statistics and code are usually a safer starting point than individual-level genotype or phenotype data.

## 6. Try it yourself

1. Delete `sumstats_chr2.tsv` and run the agent again. Does the report change?
2. Change the goal to ask for hits below `1e-5` instead of genome-wide significance.
3. Add a new tool, for example `check_columns(filename)`, that confirms a file has the columns `SNP`, `CHR`, `BP`, `BETA` and `P`. Add it to both `TOOL_FUNCTIONS` and `TOOLS`.
4. Remove the descriptions from `TOOLS` (set them to an empty string) and see how the agent's choices change.

To skip writing the loop yourself, the Anthropic Python SDK also has a Tool Runner that runs this round trip for you: https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner